In [0]:
%sql

CREATE OR REPLACE FUNCTION admin.access_control.row_filter_by_hierarchy(target_customer_id STRING)
RETURN 
  EXISTS (
    -- Step 1: Identify the User and their Role based on the session user
    WITH user_context AS (
        SELECT
            employee_name,
            LOWER(TRIM(role)) as role_name
        FROM admin.access_control.sales_reporting_hierarchy
        WHERE LOWER(TRIM(email)) = LOWER(TRIM(current_user()))
    ),
    -- Check staff-customer mapping
    access_check AS (
        SELECT 1
        FROM user_context uc
        LEFT JOIN admin.access_control.customer_manager cm
            ON CAST(cm.customer_code AS STRING) = CAST(target_customer_id AS STRING)
        LEFT JOIN admin.access_control.sales_reporting_hierarchy srh_csm 
            ON LOWER(TRIM(cm.customer_success_manager_name)) = LOWER(TRIM(srh_csm.employee_name))
        LEFT JOIN admin.access_control.sales_reporting_hierarchy srh_lead
            ON LOWER(TRIM(srh_csm.manager_name)) = LOWER(TRIM(srh_lead.employee_name))
        WHERE
            -- Director 
            uc.role_name IN ('sales director', 'director')
            -- CSM 
            OR LOWER(TRIM(uc.employee_name)) = LOWER(TRIM(cm.customer_success_manager_name))
            -- Team Lead
            OR LOWER(TRIM(uc.employee_name)) = LOWER(TRIM(srh_csm.manager_name))
            -- AD
            OR LOWER(TRIM(uc.employee_name)) = LOWER(TRIM(srh_lead.manager_name))
    )
    SELECT 1 FROM access_check
  );

In [0]:
%sql
ALTER TABLE customer.customer_information.mart_customer_spending_summary
SET ROW FILTER admin.access_control.row_filter_by_hierarchy ON (customer_id);